# Kateto 1.5B v2 — SFT con datos normalizados + estados de voz (2xT4)

Objetivo: un 1.5B **coherente**, que conteste y no se esconda detras del centinela de silencio.

Que cambia respecto de la corrida local (que vio 1500 muestras por epoca y ctx 256):

- **Dataset v2** (36202 muestras): todas con el formato exacto del motor de inferencia
  (`<|im_user|>...<|im_end|>\n<|im_start|>seco\n...<|im_end|>`). El mix viejo tenia 17673
  muestras (45%) sin cerrar el turno de usuario.
- **Silencios rebalanceados**: de 3916 muestras con `<|no_response|>` se conservan 783 (20%).
- **ctx 1024** (local era 256) y **epocas completas de 36000 muestras** (local: 1500).
- **Estados de voz del 1.5B** (Capa 2, voz `seco`) para que el motor tenga su marcador.

ORPO queda **fuera** de esta corrida a proposito: la implementacion vendoreada en
`RWKV-PEFT` tiene dos defectos medidos (off-by-one de `ctx_len` y `nan` en el paso 2),
esta en arreglo aparte. La palanca de coherencia de hoy es el dato + las epocas + el ctx.

In [ ]:
# CELL 1 — Setup: env, deps, GPU probe
import os
os.environ["NCCL_P2P_DISABLE"] = "1"   # 2xT4 sin NVLink: DDP sobre PCIe
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["RWKV_MY_TESTING"] = "x070"
os.environ["WKV"] = "fla"

!pip uninstall -y torchao
!pip install -q --no-cache-dir rwkv rwkv-fla einops jsonlines peft datasets triton huggingface_hub "lightning>=2.0.0"
!nvidia-smi

In [ ]:
# CELL 2 — Modelo base 1.5B (3 GB desde HF)
import os, shutil, torch
from huggingface_hub import hf_hub_download

MODEL_REPO = "BlinkDL/rwkv7-g1"
MODEL_FILE = "rwkv7-g1j-1.5b-20260831-ctx16384.pth"
N_LAYER, N_EMBD = 24, 2048

os.makedirs("/kaggle/working/models", exist_ok=True)
model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir="/kaggle/working/models")
if os.path.islink(model_path):
    target = os.path.realpath(model_path)
    os.remove(model_path)
    shutil.copy2(target, model_path)
shutil.rmtree(os.path.expanduser("~/.cache/huggingface"), ignore_errors=True)
print("base:", model_path, os.path.getsize(model_path), "bytes")
print("GPUs:", torch.cuda.device_count())

In [ ]:
# CELL 3 — Stage del dataset desde /kaggle/input (layout agnostico)
import os, glob, shutil, zipfile

INPUT = "/kaggle/input"
# El layout cambia segun el entorno: /kaggle/input/<slug>, /kaggle/input/datasets/<owner>/<slug>,
# o symlinks. Caminamos con followlinks para no quedarnos afuera, y mostramos el arbol.
print("arbol de /kaggle/input:")
vistos = 0
for root, dirs, files in os.walk(INPUT, followlinks=True):
    prof = root.replace(INPUT, "").count("/")
    if prof > 3:
        dirs[:] = []
        continue
    for n in sorted(files)[:8]:
        print("   ", os.path.join(root, n))
        vistos += 1
    for d in sorted(dirs)[:8]:
        print("   ", os.path.join(root, d) + "/")
    if vistos > 60:
        break

zips = sorted(glob.glob(f"{INPUT}/**/*.zip", recursive=True))
v2s = sorted(glob.glob(f"{INPUT}/**/rwkv_kateto_v2_train.jsonl", recursive=True))
pefts = sorted(glob.glob(f"{INPUT}/**/RWKV-PEFT/train.py", recursive=True))
print("zips:", zips[:10])
print("v2 jsonl:", v2s[:3])
print("RWKV-PEFT/train.py:", pefts[:3])
assert zips or (v2s and pefts), "no encontre el dataset de Kateto montado en /kaggle/input"

TARGET = "/kaggle/working/kateto-train"
os.makedirs(TARGET, exist_ok=True)

for z in zips:
    base = os.path.basename(z)[:-4]
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
    top = names[0].split("/")[0] if names else ""
    dest = TARGET if top == base else os.path.join(TARGET, base)
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dest)
    print(f"unzip {os.path.basename(z)} -> {dest} ({len(names)} entradas)")

# carpetas sueltas (Kaggle ya descomprime los zip de los datasets)
if not os.path.exists(f"{TARGET}/RWKV-PEFT/train.py"):
    src = os.path.dirname(os.path.dirname(pefts[0])) if pefts else None
    if src and os.path.exists(os.path.join(src, "data")):
        print("copiando desde:", src)
        for p in glob.glob(f"{src}/**/*", recursive=True):
            if os.path.isfile(p):
                rel = os.path.relpath(p, src)
                dst = os.path.join(TARGET, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                if not os.path.exists(dst):
                    shutil.copy2(p, dst)

os.makedirs("/kaggle/working/out", exist_ok=True)
if not os.path.exists(f"{TARGET}/out"):
    os.symlink("/kaggle/working/out", f"{TARGET}/out")

for req in ["RWKV-PEFT/train.py", "data/rwkv_kateto_v2_train.jsonl", "data/orpo_silencio.jsonl",
            "data/eval_prompts.jsonl", "data/rwkv_kateto_seco.jsonl", "rwkv_pipeline/infer_kateto.py",
            "rwkv_pipeline/train_state_voice.sh", "RWKV-PEFT/rwkvt/dataset/dataset.py"]:
    assert os.path.exists(f"{TARGET}/{req}"), f"falta {req}"
    print("ok:", req)
print("jsonl en data/:", len(glob.glob(f"{TARGET}/data/*.jsonl")))

In [ ]:
# CELL 4 — Sanidad de datos: formato exacto y pares ORPO utilizables
import json, re

TARGET = "/kaggle/working/kateto-train"
V2 = f"{TARGET}/data/rwkv_kateto_v2_train.jsonl"
ORPO = f"{TARGET}/data/orpo_silencio.jsonl"
TURNO = re.compile(r"^(?:<\|im_user\|>[^<]*(?:<(?!\|im_end\|>)[^<]*)*<\|im_end\|>\n"
                   r"<\|im_start\|>[A-Za-z0-9_]+\n[^<]*(?:<(?!\|im_end\|>)[^<]*)*<\|im_end\|>\n?)+$")

ok = mal = 0
for line in open(V2, encoding="utf-8"):
    t = json.loads(line)["text"]
    if TURNO.match(t):
        ok += 1
    else:
        mal += 1
print(f"train v2: {ok} bien formadas, {mal} mal formadas")
assert mal == 0, "hay muestras mal formadas"

# ORPO: nos quedamos con los pares cuyo ELEGIDO es una respuesta real (sin centinelas).
# Si el dataset trae algunos sucios, se filtran aca y se informa: no vale tirar una corrida de 5 h.
limpios, sucios = [], 0
for line in open(ORPO, encoding="utf-8"):
    d = json.loads(line)
    assert set(d) == {"prompt", "chosen", "rejected"}, d.keys()
    assert d["rejected"].startswith("<|no_response|>")
    if "<|no_response|>" in d["chosen"] or "<|wait|>" in d["chosen"]:
        sucios += 1
        continue
    limpios.append(d)
print(f"orpo: {len(limpios)} pares usables, {sucios} descartados por traer centinela en el elegido")
with open(ORPO, "w", encoding="utf-8") as fh:
    for d in limpios:
        fh.write(json.dumps(d, ensure_ascii=False) + "\n")
assert len(limpios) > 1000, "quedaron muy pocos pares ORPO"
print("eval prompts:", sum(1 for _ in open(f"{TARGET}/data/eval_prompts.jsonl", encoding="utf-8")))

In [ ]:
# CELL 5 — SONDA v2: precision y tamano de lote. En T4 (Turing) bf16 no tiene tensor cores
# nativos: se emula. La v1 midio 5,09 s/paso a ctx1024/micro4 y 2,75 s/paso a ctx512/micro8 (bf16).
# Aca se mide fp16 vs bf16 y micro4 vs micro8 a ctx 512, que es donde el dataset vive comodo.
import os, subprocess, time, torch

TARGET = "/kaggle/working/kateto-train"
CWD = f"{TARGET}/RWKV-PEFT"
MODEL = "/kaggle/working/models/rwkv7-g1j-1.5b-20260831-ctx16384.pth"
n_gpu = torch.cuda.device_count()
devices, strategy = (str(n_gpu), "ddp") if n_gpu > 1 else ("1", "auto")
print("GPUs:", n_gpu, "->", devices, strategy, flush=True)


def probar(etiqueta, ctx, micro, precision, steps=60):
    proj = f"/kaggle/working/out/probe2_{etiqueta}"
    os.makedirs(proj, exist_ok=True)
    cmd = ["python", "train.py",
           "--load_model", MODEL, "--proj_dir", proj,
           "--data_file", f"{TARGET}/data/rwkv_kateto_v2_train.jsonl",
           "--data_type", "jsonl", "--loss_mask", "qa", "--vocab_size", "65536",
           "--n_layer", "24", "--n_embd", "2048",
           "--ctx_len", str(ctx), "--micro_bsz", str(micro),
           "--epoch_steps", str(steps), "--epoch_count", "1", "--epoch_save", "1",
           "--lr_init", "2e-4", "--lr_final", "2e-5",
           "--accelerator", "gpu", "--devices", devices, "--strategy", strategy,
           "--precision", precision, "--grad_cp", "1",
           "--my_testing", "x070", "--op", "fla",
           "--peft", "pissa", "--pissa_config", '{"pissa_load":"","pissa_init":"","pissa_r":32,"svd_niter":4}',
           "--train_parts", '["time","ln"]', "--merge", "0", "--num_workers", "2"]
    log = f"/kaggle/working/out/probe2_{etiqueta}.log"
    t0 = time.time()
    with open(log, "w") as fh:
        subprocess.run(cmd, cwd=CWD, stdout=fh, stderr=subprocess.STDOUT)
    dt = time.time() - t0
    spp = dt / steps
    muestras_h = (3600 / spp) * micro if spp else 0
    print(f"== {etiqueta}: ctx={ctx} micro={micro}x{n_gpu} prec={precision} steps={steps} "
          f"wall={dt:.0f}s => {spp:.2f} s/paso", flush=True)
    print(f"   muestras/hora: {muestras_h:.0f} | muestras que entran en 10,5 h: {int(muestras_h*10.5)} "
          f"(el dataset tiene 36202)", flush=True)
    print(f"   pasos en 10,5 h: {int(10.5*3600/spp)}", flush=True)
    return spp


for etiqueta, ctx, micro, prec in [
        ("c512_m4_fp16", 512, 4, "fp16"),
        ("c512_m4_bf16", 512, 4, "bf16"),
        ("c512_m8_fp16", 512, 8, "fp16"),
        ("c1024_m4_fp16", 1024, 4, "fp16")]:
    probar(etiqueta, ctx, micro, prec)
    print("", flush=True)

print("=== FIN SONDA v2 ===", flush=True)
